# 3 - Encoding — categóricas para formato numérico

## Objetivo

Converter as features categóricas em representações numéricas que a GAIN consiga consumir, preservando a possibilidade de inversão (etapas 4 e 6).

## O que é codificado

1. **`Codigo Local`** (8 estações) → one-hot de 8 colunas `est_<codigo>`.
2. **`<Var>_LD`** apenas para variáveis com `decisao ∈ {moderada, alta}` em `lds_resumo.csv` — **6** variáveis: Nitrato, DBO, Nitrogênio Amoniacal Total, Coliformes Termotolerantes, Cianobactérias, Microcistinas. Cada uma vira 3 colunas `<Var>_LD_{normal, lt, gt}`.
3. **`umido`** já é binária (gerada em `02_features_temporais`); passa direto, sem one-hot.

As 7 variáveis com `decisao ∈ {sem_censura, desprezivel}` têm a coluna `_LD` **descartada** (Fósforo Total, Condutividade, pH, Turbidez, Temperatura da Água, Sólidos Suspensos Totais, OD).

## One-hot vs embedding para `Codigo Local`

Com 8 categorias, one-hot é tratável (8 colunas) e mantém interpretabilidade direta. Embedding faria sentido com dezenas/centenas de categorias. **Reavaliar** na Etapa 4 se o gerador subajustar a especialização por estação — registrado em `Code/.../memory/decisoes_eda_qualiagua.md`.

## Caso especial — Coliformes Termotolerantes

Única variável com censura predominante à **direita** (`>` = 28 ocorrências vs `<` = 2). Reflete o **teto reportável** do método de tubos múltiplos (saturação superior), não piso analítico. As 3 colunas geradas (`_LD_normal`, `_LD_lt`, `_LD_gt`) cobrem ambos os casos.

## Setup

Imports + caminhos relativos a `Code/3 - Preprocessing/03_encoding.ipynb`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

IN_PARQUET   = Path("../../Data/ProcessedData/dataset_com_tempo.parquet")
LDS_RESUMO   = Path("../../Data/Figures/01_EDA/tabelas/lds_resumo.csv")
OUT_PARQUET  = Path("../../Data/ProcessedData/dataset_encoded.parquet")
OUT_PKL      = Path("../../Data/ProcessedData/encoders.pkl")
OUT_JSON     = Path("../../Data/ProcessedData/encoded_columns.json")

# Ordem fixa para reprodutibilidade do one-hot
STATIONS = ["CM320", "JC341", "JC342", "MR361", "MR363", "MR369", "TJ303", "TJ306"]
LD_CATEGORIES = ["normal", "lt", "gt"]  # NaN→normal, '<'→lt, '>'→gt

VARS = [
    "DBO", "OD", "Nitrato", "Nitrogênio Amoniacal Total", "Fósforo Total",
    "Condutividade", "pH", "Turbidez", "Temperatura da Água",
    "Sólidos Suspensos Totais", "Coliformes Termotolerantes",
    "Cianobacterias", "Microcistinas",
]

TEMPORAIS_GAIN = ["ano_norm", "Mes_sin", "Mes_cos", "umido", "dias_desde_inicio"]
IDENTIFICADORES = ["Data", "Ano_int"]

## Carregamento

Lê o dataset já com features temporais e a tabela de decisões de censura. Filtra as variáveis cuja `_LD` deve ser codificada.

In [2]:
df = pd.read_parquet(IN_PARQUET)
print(f"Shape de entrada: {df.shape}")

tab_lds = pd.read_csv(LDS_RESUMO, encoding="utf-8").set_index("variavel")

vars_ld_codificar = tab_lds[tab_lds["decisao"].isin(["moderada", "alta"])].index.tolist()
vars_ld_descartar = tab_lds[~tab_lds["decisao"].isin(["moderada", "alta"])].index.tolist()

print(f"\n_LD a codificar ({len(vars_ld_codificar)}): {vars_ld_codificar}")
print(f"_LD a descartar ({len(vars_ld_descartar)}): {vars_ld_descartar}")

assert len(vars_ld_codificar) == 6, "Esperado 6 variáveis para codificar _LD; revisar lds_resumo.csv."
assert set(vars_ld_codificar) <= set(VARS), "Variáveis fora da lista canônica."

Shape de entrada: (657, 37)

_LD a codificar (6): ['Nitrato', 'Cianobacterias', 'Microcistinas', 'Coliformes Termotolerantes', 'DBO', 'Nitrogênio Amoniacal Total']
_LD a descartar (7): ['OD', 'Fósforo Total', 'Condutividade', 'Temperatura da Água', 'Turbidez', 'pH', 'Sólidos Suspensos Totais']


## 1. One-hot de `Codigo Local`

`OneHotEncoder` com `categories=[STATIONS]` para forçar a ordem (reprodutibilidade) e `handle_unknown='error'` para falhar barulhentamente se aparecer uma estação fora do conjunto previsto. As colunas seguem o padrão `est_<codigo>` para facilitar leitura.

In [3]:
enc_estacao = OneHotEncoder(
    categories=[STATIONS],
    handle_unknown="error",
    sparse_output=False,
    dtype=np.int64,
)

X_estacao = enc_estacao.fit_transform(df[["Codigo Local"]])
cols_estacao = [f"est_{c}" for c in STATIONS]
df_estacao = pd.DataFrame(X_estacao, columns=cols_estacao, index=df.index)

# Sanidade: cada linha tem exatamente um 1 (estação atribuída)
assert (df_estacao.sum(axis=1) == 1).all(), "Linha sem ou com mais de uma estação ativa."
print(f"Colunas geradas: {cols_estacao}")
print(f"\nContagem por estação:")
df_estacao.sum().rename("n_linhas").to_frame()

Colunas geradas: ['est_CM320', 'est_JC341', 'est_JC342', 'est_MR361', 'est_MR363', 'est_MR369', 'est_TJ303', 'est_TJ306']

Contagem por estação:


,n_linhas
est_CM320,114
est_JC341,10
est_JC342,114
est_MR361,115
est_MR363,8
est_MR369,115
est_TJ303,113
est_TJ306,68


## 2. One-hot dos `_LD` selecionados

Para cada uma das 6 variáveis em `decisao ∈ {moderada, alta}`:

1. Mapear `NaN → 'normal'`, `'<' → 'lt'`, `'>' → 'gt'`.
2. Aplicar `OneHotEncoder` com `categories=[LD_CATEGORIES]` para forçar as 3 colunas mesmo quando algum nível não aparece (p.ex. Nitrato não tem `>` em todo o histórico).
3. Gerar colunas `<Var>_LD_normal`, `<Var>_LD_lt`, `<Var>_LD_gt`.

Os encoders são guardados num dicionário para inversão posterior.

In [4]:
def mapear_ld(serie: pd.Series) -> pd.Series:
    """NaN→'normal', '<'→'lt', '>'→'gt'. Qualquer outro valor lança erro."""
    s = serie.copy()
    out = pd.Series("normal", index=s.index, dtype="object")
    out[s == "<"] = "lt"
    out[s == ">"] = "gt"
    inesperados = s.dropna().loc[~s.dropna().isin(["<", ">"])]
    assert inesperados.empty, f"Valores inesperados em _LD: {sorted(inesperados.unique())}"
    return out


encoders_ld: dict[str, OneHotEncoder] = {}
blocos_ld: list[pd.DataFrame] = []
resumo_ld_rows = []

for v in vars_ld_codificar:
    col_origem = f"{v}_LD"
    mapeada = mapear_ld(df[col_origem])

    enc = OneHotEncoder(
        categories=[LD_CATEGORIES],
        handle_unknown="error",
        sparse_output=False,
        dtype=np.int64,
    )
    X = enc.fit_transform(mapeada.values.reshape(-1, 1))
    cols = [f"{v}_LD_{nv}" for nv in LD_CATEGORIES]
    blocos_ld.append(pd.DataFrame(X, columns=cols, index=df.index))
    encoders_ld[v] = enc

    resumo_ld_rows.append({
        "variavel":   v,
        "n_normal":   int((mapeada == "normal").sum()),
        "n_lt":       int((mapeada == "lt").sum()),
        "n_gt":       int((mapeada == "gt").sum()),
    })

df_ld = pd.concat(blocos_ld, axis=1)

# Sanidade: soma == 1 por linha em cada bloco de 3 colunas
for v in vars_ld_codificar:
    cols_v = [f"{v}_LD_{nv}" for nv in LD_CATEGORIES]
    assert (df_ld[cols_v].sum(axis=1) == 1).all(), f"Linha sem categoria definida em {v}_LD."

pd.DataFrame(resumo_ld_rows).set_index("variavel")

,n_normal,n_lt,n_gt
variavel,,,
Nitrato,583,74,0
Cianobacterias,646,11,0
Microcistinas,653,0,4
Coliformes Termotolerantes,627,2,28
DBO,642,15,0
Nitrogênio Amoniacal Total,646,11,0


## 3. Passagem direta de `umido`

`umido` já chegou binária (`0` seco / `1` úmido) do `02_features_temporais.ipynb`. Não exige one-hot — uma feature binária com um único par `(0, 1)` é redundante de codificar como duas colunas (um teto de informação igual a uma só).

In [5]:
assert df["umido"].isin([0, 1]).all(), "umido fora de {0, 1}."
print(f"umido — distribuição: {df['umido'].value_counts().to_dict()}")

umido — distribuição: {0: 376, 1: 281}


## 4. Reorganização final

Monta o dataframe final na ordem:

1. **Identificadoras** — `Data` (datetime, auditoria), `Ano_int` (int, split temporal).
2. **Variáveis numéricas transformadas** — as 13 do dataset original (vindas do `01_transformacoes`).
3. **Features temporais derivadas** — `ano_norm`, `Mes_sin`, `Mes_cos`, `umido`, `dias_desde_inicio`.
4. **One-hot de estação** — 8 colunas `est_*`.
5. **One-hot dos `_LD`** — 6 × 3 = 18 colunas.

**Descartado:** `Local` (string, redundante com `Codigo Local`), `Ano` (redundante com `Ano_int`), `Mes` (auxiliar já consumido por `Mes_sin/cos`), `Codigo Local` (substituída pelo one-hot), os 13 `_LD` originais (6 viraram one-hot, 7 sem censura ou desprezíveis foram descartados).

In [6]:
df_final = pd.concat([df[IDENTIFICADORES + VARS + TEMPORAIS_GAIN], df_estacao, df_ld], axis=1)

ordem_colunas = (
    IDENTIFICADORES
    + VARS
    + TEMPORAIS_GAIN
    + list(df_estacao.columns)
    + list(df_ld.columns)
)
df_final = df_final[ordem_colunas]

# Sanidade: sem colunas object remanescentes (exceto Data, que é datetime64)
tipos_object = df_final.select_dtypes(include="object").columns.tolist()
assert not tipos_object, f"Colunas object remanescentes: {tipos_object}"

# Conferência das contagens esperadas
esperado = {
    "identificadores": 2,
    "numericas":       13,
    "temporais":       5,
    "estacao_onehot":  8,
    "ld_onehot":       6 * 3,
}
total_esperado = sum(esperado.values())
print(f"Colunas esperadas: {esperado} → total {total_esperado}")
print(f"Shape final: {df_final.shape}")
assert df_final.shape[1] == total_esperado, "Total de colunas difere do esperado."

Colunas esperadas: {'identificadores': 2, 'numericas': 13, 'temporais': 5, 'estacao_onehot': 8, 'ld_onehot': 18} → total 46
Shape final: (657, 46)


In [7]:
# Amostra das primeiras linhas (algumas colunas representativas)
amostra_cols = (
    ["Data", "Ano_int", "Nitrato", "ano_norm", "umido", "est_CM320", "est_TJ303",
     "Nitrato_LD_normal", "Nitrato_LD_lt", "Coliformes Termotolerantes_LD_gt"]
)
df_final[amostra_cols].head(8)

,Data,Ano_int,Nitrato,ano_norm,umido,est_CM320,est_TJ303,Nitrato_LD_normal,Nitrato_LD_lt,Coliformes Termotolerantes_LD_gt
0,2012-01-12,2012,-6.477419,0.0,1,1,0,1,0,0
1,2012-02-09,2012,-13.020340,0.0,1,1,0,0,1,0
2,2012-03-08,2012,-4.694881,0.0,1,1,0,1,0,1
3,2012-04-12,2012,-4.694881,0.0,0,1,0,1,0,0
4,2012-05-17,2012,-13.020340,0.0,0,1,0,1,0,0
5,2012-08-08,2012,-9.305474,0.0,0,1,0,1,0,0
6,2012-10-03,2012,-9.305474,0.0,0,1,0,1,0,0
7,2012-11-07,2012,0.374773,0.0,1,1,0,1,0,0


## 5. Persistência

Três artefatos em `Data/ProcessedData/`:

- `dataset_encoded.parquet` — dataframe final.
- `encoders.pkl` — dicionário `{'estacao': OneHotEncoder, 'ld': {variavel: OneHotEncoder}}` via `joblib`.
- `encoded_columns.json` — lista ordenada das colunas do parquet, segmentada por grupo. Crítico para `04_normalizacao` saber **quais colunas normalizar** (numéricas + escalares) e quais ignorar (one-hot, identificadores).

In [8]:
df_final.to_parquet(OUT_PARQUET, index=False)

joblib.dump({"estacao": enc_estacao, "ld": encoders_ld}, OUT_PKL)

encoded_columns = {
    "identificadores":   IDENTIFICADORES,
    "numericas":         VARS,
    "temporais":         TEMPORAIS_GAIN,
    "estacao_onehot":    list(df_estacao.columns),
    "ld_onehot":         list(df_ld.columns),
    "ordem_completa":    ordem_colunas,
}
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(encoded_columns, f, ensure_ascii=False, indent=2)

print(f"Salvo: {OUT_PARQUET} ({OUT_PARQUET.stat().st_size} bytes)")
print(f"Salvo: {OUT_PKL}     ({OUT_PKL.stat().st_size} bytes)")
print(f"Salvo: {OUT_JSON}    ({OUT_JSON.stat().st_size} bytes)")

Salvo: ..\..\Data\ProcessedData\dataset_encoded.parquet (56930 bytes)
Salvo: ..\..\Data\ProcessedData\encoders.pkl     (2941 bytes)
Salvo: ..\..\Data\ProcessedData\encoded_columns.json    (2391 bytes)


## Síntese final

- **Shape final:** 657 × 46 (2 ID + 13 numéricas + 5 temporais + 8 estação + 18 LD).
- **Tipos:** apenas `Data` é datetime; todas as outras 45 colunas são numéricas. Nenhum `object` remanescente.
- **Inversibilidade:** `encoders.pkl` permite reconstruir `Codigo Local` e cada `<Var>_LD` aplicando `inverse_transform`. `encoded_columns.json` documenta a ordem exata das colunas.
- **Próximo:** `04_normalizacao.ipynb` consome `dataset_encoded.parquet` + `encoded_columns.json` e aplica `MinMaxScaler(-1, 1)` apenas nas listas `numericas` e na coluna escalar `dias_desde_inicio`. As demais (one-hot, `umido`, `Mes_sin/cos`, `ano_norm`) já estão em escalas adequadas.